# makemore 生产案例集（含完整中文注释）

在前面的教学 notebook 里，我们用**裸张量**手搓了字符级语言模型。
本文件展示同一个模型在**真实项目**里通常怎么写——这些写法在公司做名字/文案/序列生成时几乎都会用到：

| 案例 | 场景 | 涉及的生产技巧 |
|------|------|----------------|
| 案例 1 | 把模型封装成 `nn.Module` 类 | 标准模型定义、参数自动管理 |
| 案例 2 | 带验证监控 + 早停 + 存最优的训练循环 | `optim.Adam`、early stopping、checkpoint |
| 案例 3 | 保存 / 加载模型用于上线推理 | `state_dict`、设备无关加载、部署 |
| 案例 4 | 可控采样：温度 + top-k + 指定前缀续写 | 控制生成多样性、条件生成 |
| 案例 5 | 批量生成"全新"结果 + 评估指标 | 去重过滤、困惑度 perplexity |

> 依赖同目录的 `names.txt`。全部单元从上往下运行即可。

## 0. 通用准备（数据 + 词表 + 设备）

所有案例共用这一段：读数据、建词表、切分训练/验证集、选设备(CPU/GPU)。

In [2]:
import torch                       # PyTorch 主库
import torch.nn as nn              # 神经网络模块(层、损失等)
import torch.nn.functional as F    # 函数式接口(softmax、cross_entropy 等)

# --- 设备无关:有 GPU 用 GPU,否则用 CPU(生产代码都这么写) ---
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')  # 依次尝试 N卡/苹果芯片/CPU
print('使用设备:', device)                          # 打印当前设备

# --- 读数据 + 建词表 ---
words = open('names.txt', 'r').read().splitlines()  # 读入名字列表
chars = sorted(list(set(''.join(words))))           # 去重排序得到所有字符
stoi = {s: i + 1 for i, s in enumerate(chars)}      # 字符->编号,1~26
stoi['.'] = 0                                        # '.' 作为开始/结束符,编号 0
itos = {i: s for s, i in stoi.items()}              # 编号->字符
vocab_size = len(itos)                              # 词表大小 = 27
block_size = 3                                       # 上下文长度:用前 3 个字符预测下一个
print('词表大小:', vocab_size, '  样本名字数:', len(words))  # 打印基本信息

# --- 把名字列表转成 (X, Y) 张量数据 ---
def build_dataset(words):                            # 输入名字列表,输出(输入上下文, 目标字符)
    X, Y = [], []                                    # X=上下文, Y=目标
    for w in words:                                  # 遍历名字
        context = [0] * block_size                   # 初始上下文全是 '.'
        for ch in w + '.':                          # 遍历字符(末尾补 '.')
            X.append(context)                        # 记录上下文
            Y.append(stoi[ch])                       # 记录目标字符
            context = context[1:] + [stoi[ch]]      # 滑动窗口更新上下文
    return torch.tensor(X), torch.tensor(Y)          # 转成张量返回

import random                                        # 用于打乱
random.seed(42)                                      # 固定随机种子
random.shuffle(words)                                # 打乱名字顺序
n1 = int(0.9 * len(words))                           # 90% 训练, 10% 验证
Xtr, Ytr = build_dataset(words[:n1])                 # 训练集
Xdev, Ydev = build_dataset(words[n1:])               # 验证集
Xtr, Ytr = Xtr.to(device), Ytr.to(device)            # 数据搬到目标设备
Xdev, Ydev = Xdev.to(device), Ydev.to(device)        # 同上
print('训练样本:', Xtr.shape[0], ' 验证样本:', Xdev.shape[0])  # 打印数量

使用设备: mps
词表大小: 27   样本名字数: 32033
训练样本: 205280  验证样本: 22866


## 案例 1：把模型封装成 `nn.Module` 类

生产里几乎不用裸张量手写前向，而是继承 `nn.Module`：层和参数由框架自动管理，方便保存、迁移设备、组合。

In [5]:
class MakemoreMLP(nn.Module):                                   # 继承 nn.Module 定义模型
    def __init__(self, vocab_size, block_size, n_embd=10, n_hidden=200):  # 构造函数:传入超参数
        super().__init__()                                      # 必须先调用父类构造
        self.block_size = block_size                            # 记住上下文长度(采样时要用)
        self.emb = nn.Embedding(vocab_size, n_embd)             # 嵌入层:把字符编号映射成向量
        self.fc1 = nn.Linear(n_embd * block_size, n_hidden)     # 全连接隐藏层
        self.fc2 = nn.Linear(n_hidden, vocab_size)              # 输出层:得到每个字符的得分

    def forward(self, x):                                       # 前向传播:x 形状 (批大小, block_size)
        emb = self.emb(x)                                       # 查嵌入 -> (批, block_size, n_embd)
        emb = emb.view(emb.shape[0], -1)                        # 展平成 (批, block_size*n_embd)
        h = torch.tanh(self.fc1(emb))                           # 过隐藏层 + tanh 激活
        logits = self.fc2(h)                                    # 输出层得到 logits (批, vocab_size)
        return logits                                           # 返回得分(不在这里做 softmax)

torch.manual_seed(42)                                           # 固定随机种子,保证可复现
model = MakemoreMLP(vocab_size, block_size).to(device)          # 实例化模型并搬到设备
print(model)                                                    # 打印模型结构
print('参数总量:', sum(p.numel() for p in model.parameters()))  # 打印可训练参数个数

MakemoreMLP(
  (emb): Embedding(27, 10)
  (fc1): Linear(in_features=30, out_features=200, bias=True)
  (fc2): Linear(in_features=200, out_features=27, bias=True)
)
参数总量: 11897


## 案例 2：带验证监控 + 早停 + 保存最优的训练循环

这是**生产标准训练流程**：用 Adam 优化器；每隔若干步在验证集上评估；验证 loss 不再下降就**早停**并只保留**表现最好**的那份权重。

In [6]:
@torch.no_grad()                                    # 评估时不追踪梯度(更快省显存)
def eval_loss(model, X, Y, batch=4096):             # 分批计算整个数据集上的平均 loss
    model.eval()                                    # 切到评估模式(影响 dropout/BN 等)
    total, cnt = 0.0, 0                             # 累加 loss 和样本数
    for i in range(0, X.shape[0], batch):           # 分批遍历,避免一次性吃爆显存
        xb, yb = X[i:i+batch], Y[i:i+batch]         # 取一批
        loss = F.cross_entropy(model(xb), yb, reduction='sum')  # 该批 loss 之和
        total += loss.item()
        cnt += xb.shape[0]    # 累加
    model.train()                                   # 切回训练模式
    return total / cnt                              # 返回平均 loss

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # Adam 优化器(生产最常用)

max_steps   = 5000        # 最多训练步数
batch_size  = 64          # 每步 mini-batch 大小
eval_every  = 500         # 每隔多少步评估一次验证集
patience    = 3           # 早停:验证 loss 连续这么多次没改善就停
best_val    = float('inf')# 记录最优验证 loss
bad_count   = 0           # 连续没改善的次数
best_state  = None        # 保存最优时刻的权重快照

for step in range(1, max_steps + 1):                             # 训练主循环
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), device=device)  # 随机抽一批下标
    xb, yb = Xtr[ix], Ytr[ix]                                    # 取这批数据
    logits = model(xb)                                           # 前向
    loss = F.cross_entropy(logits, yb)                           # 交叉熵 loss
    optimizer.zero_grad()                                        # 清空上一步梯度
    loss.backward()                                              # 反向传播
    optimizer.step()                                             # 更新参数

    if step % eval_every == 0:                                   # 到评估点
        val = eval_loss(model, Xdev, Ydev)                       # 算验证集 loss
        print(f'step {step:5d} | train {loss.item():.4f} | val {val:.4f}')  # 打印
        if val < best_val - 1e-4:                                # 验证 loss 有明显改善
            best_val = val                                       # 更新最优
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}  # 快照最优权重
            bad_count = 0                                        # 重置计数
        else:                                                    # 没改善
            bad_count += 1                                       # 计数 +1
            if bad_count >= patience:                            # 达到耐心上限
                print(f'验证 loss 连续 {patience} 次未改善,提前停止'); break  # 早停

if best_state is not None:                                       # 训练结束
    model.load_state_dict(best_state)                            # 载回表现最好的那份权重
print('最优验证 loss:', round(best_val, 4))                      # 打印最优结果

step   500 | train 2.5884 | val 2.4372
step  1000 | train 2.2727 | val 2.3723
step  1500 | train 2.2337 | val 2.3398
step  2000 | train 2.3822 | val 2.3111
step  2500 | train 2.1294 | val 2.2951
step  3000 | train 2.2235 | val 2.2788
step  3500 | train 2.3264 | val 2.2717
step  4000 | train 2.1878 | val 2.2605
step  4500 | train 2.0859 | val 2.2476
step  5000 | train 2.2282 | val 2.2389
最优验证 loss: 2.2389


## 案例 3：保存 / 加载模型用于上线推理

上线时不会带着训练代码，而是保存**权重 + 词表 + 超参数**，服务端加载后即可推理。
`map_location=device` 让 GPU 上训练的模型也能在 CPU 机器上加载。

In [8]:
CKPT = 'makemore_model.pt'                          # checkpoint 文件名

# --- 保存:权重 + 词表 + 结构超参,一起存(推理端需要它们才能重建模型) ---
torch.save({
    'state_dict': model.state_dict(),               # 模型权重
    'stoi': stoi, 'itos': itos,                      # 词表映射
    'vocab_size': vocab_size, 'block_size': block_size,  # 结构超参
}, CKPT)                                             # 写入文件
print('已保存到', CKPT)                              # 提示

# --- 加载:模拟服务端从零重建模型 ---
ckpt = torch.load(CKPT, map_location=device)         # 读 checkpoint(设备无关)
infer_model = MakemoreMLP(ckpt['vocab_size'], ckpt['block_size']).to(device)  # 按超参重建空模型
infer_model.load_state_dict(ckpt['state_dict'])      # 灌入权重
infer_model.eval()                                   # 切评估模式(推理必做)
itos_loaded = ckpt['itos']                           # 取回词表
print('加载完成,可用于推理')                          # 提示

已保存到 makemore_model.pt
加载完成,可用于推理


## 案例 4：可控采样——温度 + top-k + 指定前缀续写

生产里生成文本几乎都要**可控**：
- **温度 temperature**：<1 更保守稳重，>1 更随机有创意。
- **top-k**：每步只在概率最高的 k 个候选里采样，过滤掉长尾噪声。
- **前缀 prefix**：给定开头字母，让模型**续写**（条件生成）。

In [10]:
@torch.no_grad()                                                     # 推理不需要梯度
def generate(model, itos, n=10, temperature=1.0, top_k=None, prefix='', seed=None):  # 通用生成函数
    model.eval()                                                     # 评估模式
    stoi_local = {s: i for i, s in itos.items()}                     # 由 itos 反推 stoi
    g = torch.Generator(device=device)                               # 独立随机数发生器
    if seed is not None: g.manual_seed(seed)                         # 需要复现就固定种子
    results = []                                                     # 存生成结果
    for _ in range(n):                                               # 生成 n 个
        context = [0] * model.block_size                             # 初始上下文全是 '.'
        out = []                                                     # 当前样本的字符
        for ch in prefix:                                            # 先把前缀喂进去(条件生成)
            ix = stoi_local[ch]                                      # 前缀字符编号
            out.append(ix)                                           # 记录
            context = context[1:] + [ix]                             # 更新上下文
        while True:                                                  # 继续自由生成
            x = torch.tensor([context], device=device)               # 当前上下文 (1, block_size)
            logits = model(x)                                        # 前向得分
            logits = logits / temperature                            # 温度缩放:调节随机性
            if top_k is not None:                                    # 若启用 top-k
                v, _ = torch.topk(logits, top_k)                     # 取最大的 k 个得分
                logits[logits < v[:, [-1]]] = -float('inf')          # 其余置为 -inf(概率归零)
            probs = F.softmax(logits, dim=1)                         # 转成概率
            ix = torch.multinomial(probs, 1, generator=g).item()     # 按概率采样下一个字符
            if ix == 0: break                                        # 抽到 '.' 结束
            out.append(ix)                                           # 记录字符
            context = context[1:] + [ix]                             # 更新上下文
        results.append(''.join(itos[i] for i in out))               # 拼成字符串
    return results                                                   # 返回生成列表

print('温度=1.0 (默认):     ', generate(infer_model, itos_loaded, 5, seed=1))          # 常规
print('温度=0.5 (更稳重):   ', generate(infer_model, itos_loaded, 5, temperature=0.5, seed=1))  # 更保守
print('温度=1.3 (更狂野):   ', generate(infer_model, itos_loaded, 5, temperature=1.3, seed=1))  # 更随机
print('top_k=5 (只选高概率):', generate(infer_model, itos_loaded, 5, top_k=5, seed=1))          # 过滤长尾
print("前缀 'ka' 续写:      ", generate(infer_model, itos_loaded, 5, prefix='ka', seed=1))      # 条件生成

itos_loaded: {1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
k 是否存在: False
温度=1.0 (默认):      ['jase', 'adni', 'zendyolem', 'urisalyneelaa', 'amoae']
温度=0.5 (更稳重):    ['jase', 'adni', 'zand', 'dant', 'urie']
温度=1.3 (更狂野):    ['ovseradniliendyolem', 'urilaipeechaj', 'amoae', 'lua', 'veinalieo']
top_k=5 (只选高概率): ['jare', 'aley', 'jend', 'alay', 'kaila']
前缀 'ka' 续写:       ['kai', 'kassuadni', 'kazendyolem', 'kaurisalyneelaa', 'kaamoae']


## 案例 5：批量生成“全新”结果 + 评估指标
~~
~~真实场景常见需求：**批量生成大量候选，去重、过滤掉训练集里已存在的**，只保留模型“原创”的。
同时用**困惑度 perplexity**（= exp(loss)）作为可读性更强的评估指标。

In [ ]:
# --- 批量生成并只保留'全新'的名字 ---
existing = set(words)                                             # 训练数据里已有的名字集合
raw = generate(infer_model, itos_loaded, n=200, top_k=10, seed=7) # 批量生成 200 个候选
fresh = []                                                        # 存'全新'的名字
seen = set()                                                      # 生成内部也去重
for name in raw:                                                  # 遍历候选
    if len(name) < 2: continue                                    # 过滤太短的
    if name in existing: continue                                 # 过滤训练集已有的
    if name in seen: continue                                     # 过滤重复生成的
    seen.add(name); fresh.append(name)                            # 保留
print(f'200 个候选里,全新且不重复的有 {len(fresh)} 个,示例:')       # 统计
print(fresh[:15])                                                 # 打印前 15 个

# --- 评估:困惑度 perplexity(越低越好,直观表示模型'有多困惑') ---
import math                                                       # 数学库
val_loss = eval_loss(infer_model, Xdev, Ydev)                     # 验证集平均交叉熵 loss
perplexity = math.exp(val_loss)                                   # 困惑度 = e^loss
print(f'验证集 loss={val_loss:.4f}  perplexity={perplexity:.2f}') # 打印指标

## 小结：这些就是把玩具模型变成"能上线"的关键差异

| 玩具写法 | 生产写法（本文件） |
|----------|--------------------|
| 裸张量手写前向 | `nn.Module` 类，参数自动管理（案例 1）|
| 训练固定步数 | 验证监控 + 早停 + 存最优（案例 2）|
| 变量留在内存 | `state_dict` 保存/加载，设备无关（案例 3）|
| 一种采样方式 | 温度 / top-k / 前缀条件生成（案例 4）|
| 直接输出 | 去重过滤 + perplexity 指标（案例 5）|

想进一步逼近真实大模型：把 `MakemoreMLP` 换成 RNN / Transformer，加 dropout、学习率调度、混合精度(`torch.cuda.amp`)与更大的 `block_size` 即可。
